In [4]:
import os
from pathlib import Path
import requests
import pandas as pd

from utils import fetch_socrata_dataset

# Ridership data

In [5]:
# If the CTA ridership data exists, read it in
if Path("output/cta_ridership.parquet").is_file():
    cta_df = pd.read_parquet("output/cta_ridership.parquet")
else:
    # If the CTA data does not exist, download it from Socrata
    cta_df = fetch_socrata_dataset('5neh-572f')

    # And save as a flat file
    os.makedirs("output", exist_ok=True)
    cta_df.to_parquet(path='output/cta_ridership.parquet', engine='fastparquet', index=False)

In [6]:
cta_df.tail(10)

,station_id,stationname,date,daytype,rides
1285285,41480,Western-Brown,2025-08-31T00:00:00.000,U,1340
1285286,41490,Harrison,2025-08-31T00:00:00.000,U,2756
1285287,41500,Montrose-Brown,2025-08-31T00:00:00.000,U,1022
1285288,41510,Morgan-Lake,2025-08-31T00:00:00.000,U,3702
1285289,41660,Lake/State,2025-08-31T00:00:00.000,U,10951
1285290,41670,Conservatory,2025-08-31T00:00:00.000,U,558
1285291,41680,Oakton-Skokie,2025-08-31T00:00:00.000,U,250
1285292,41690,Cermak-McCormick Place,2025-08-31T00:00:00.000,U,1459
1285293,41700,Washington/Wabash,2025-08-31T00:00:00.000,U,6586
1285294,41710,Damen-Lake,2025-08-31T00:00:00.000,U,659


In [7]:
# Check that the file is up-to-date
# If the data exists and the last row number is smaller than the last row number on Socrata, re-download
nrow_in_data = cta_df.shape[0]
print(f'Number of rows in the data: {nrow_in_data}')

# Check against Socrata
url = "https://data.cityofchicago.org/resource/5neh-572f.json"
params = {
    "$select": "count(*)"
}

data_socrata_json = requests.get(url, params=params).json()
nrow_in_socrata = int(data_socrata_json[0]['count'])

if nrow_in_data == nrow_in_socrata:
    print('The local data is up-to-date.')
else:
    print('Downloading new data...')
    params_download = {
        "$offset": nrow_in_data
    }

    data_new = requests.get(url, params=params_download).json()
    df_new = pd.DataFrame(data_new)
    cta_df = pd.concat([cta_df, df_new], ignore_index=True)

    # Save the flat file
    os.makedirs("output", exist_ok=True)
    cta_df.to_parquet(path='output/cta_ridership.parquet', engine='fastparquet', index=False)


Number of rows in the data: 1285295


# Station data

In [3]:
if Path("output/cta_stations.parquet").is_file():
    cta_stations_df = pd.read_parquet("output/cta_stations.parquet")
else:
    cta_stations_df = fetch_socrata_dataset('8pix-ypme')

    # And save as a flat file
    os.makedirs("output", exist_ok=True)
    cta_stations_df.to_parquet(path='output/cta_stations.parquet', engine='fastparquet', index=False)

Grabbing chunk of data...
Grabbing chunk of data...
